# Santander Value Prediction — Baseline Models

**Question:** How accurately can we predict transaction value without using any features?

Compare constant mean, median, and log-mean predictions on a reproducible 80/20 split (`random_state=42`) using RMSLE. This holdout was used during development and is not an untouched final test.

## 1. Load data and reproduce the split

Source: [OpenML 42572](https://www.openml.org/search?id=42572&type=data).

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import train_test_split


In [2]:
dataset = fetch_openml(
    data_id=42572,
    target_column="target",
    as_frame=True,
)

X = dataset.data
y = dataset.target

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (4459, 4991)
y shape: (4459,)


In [3]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)


X_train: (3567, 4991)
X_val: (892, 4991)
y_train: (3567,)
y_val: (892,)


## 2. Mean and median baselines

Mean minimizes squared error in original units; median minimizes absolute error. Neither directly optimizes RMSLE.

In [4]:
mean_prediction = y_train.mean()
median_prediction = y_train.median()

print("Mean:", mean_prediction)
print("Median:", median_prediction)


Mean: 6031730.960521447
Median: 2256000.0


In [5]:
pred_mean = np.full(len(y_val), mean_prediction)
pred_median = np.full(len(y_val), median_prediction)

rmsle_mean = root_mean_squared_log_error(y_val, pred_mean)
rmsle_median = root_mean_squared_log_error(y_val, pred_median)

print(f"Mean Baseline RMSLE: {rmsle_mean:.4f}")
print(f"Median Baseline RMSLE: {rmsle_median:.4f}")


Mean Baseline RMSLE: 2.0351
Median Baseline RMSLE: 1.7021


## 3. Log-mean baseline

The constant minimizing training squared error on the RMSLE scale is `expm1(mean(log1p(y_train)))`.

In [6]:
log_mean_prediction = np.expm1(np.log1p(y_train).mean())
pred_log_mean = np.full(len(y_val), log_mean_prediction)

rmsle_log_mean = root_mean_squared_log_error(y_val, pred_log_mean)

print(f"Log-Mean prediction: {log_mean_prediction:.2f}")
print(f"Log-Mean Baseline RMSLE: {rmsle_log_mean:.4f}")


Log-Mean prediction: 1964498.59
Log-Mean Baseline RMSLE: 1.6963


## 4. Comparison

In [7]:
baseline_results = pd.DataFrame({
    "Model": ["Mean", "Median", "Log-Mean"],
    "Validation RMSLE": [rmsle_mean, rmsle_median, rmsle_log_mean],
})

display(baseline_results)


,Model,Validation RMSLE
0,Mean,2.035067
1,Median,1.702119
2,Log-Mean,1.696279


**Result:** Holdout RMSLE — mean **2.0351**, median **1.7021**, log-mean **1.6963**.

**Decision:** Use log-mean as the constant reference. Recalculate it within each training fold for fair CV comparisons; holdout results are exploratory, not a final test score.